<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/03_annotation_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Annotation Validation and Dataset Preparation

This notebook:
- validates RDD2022 YOLO annotations;
- validates manually created Pascal VOC annotations for SDNET2018 and Kenyan_Data;
- maps source labels to the three project classes;
- creates a normalized annotation manifest;
- creates train/validation/test splits;
- keeps the original Raw Data untouched.

**Important:** Run Notebook 02 first and confirm the observed RDD2022 class IDs.

In [14]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import random
import shutil
import hashlib
import xml.etree.ElementTree as ET
import cv2

import numpy as np
import pandas as pd
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"

RAW_DIR = DATA_DIR / "Raw Data"
SELECTED_DIR = DATA_DIR / "Selected Data"
ANNOTATED_DIR = DATA_DIR / "Annotated Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

# ---------------------------------------------------------
# Dataset locations
# ---------------------------------------------------------

# Use the SELECTED RDD2022 dataset, not the full raw dataset.
RDD_SELECTED_DIR = SELECTED_DIR / "RDD2022"
RDD_SELECTED_IMAGES = RDD_SELECTED_DIR / "images"
RDD_SELECTED_LABELS = RDD_SELECTED_DIR / "labels"

SDNET_ANNOTATED = ANNOTATED_DIR / "SDNET2018"
KENYAN_ANNOTATED = ANNOTATED_DIR / "Kenyan_Data"

SDNET_MASK_DIR = SDNET_ANNOTATED / "Labeled images"
KENYAN_MASK_DIR = KENYAN_ANNOTATED / "Labeled images"

PROCESSED_ANNOTATIONS = PROCESSED_DIR / "annotations"
PROCESSED_ANNOTATIONS.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Project target classes
# ---------------------------------------------------------

TARGET_CLASSES = {
    1: "Crack",
    2: "Pothole",
    3: "Surface deterioration",
}

# RDD2022 mapping confirmed in Notebook 02.
RDD_CLASS_ID_TO_CODE = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
    4: "OTHER",
}

RDD_CODE_TO_TARGET = {
    "D00": "Crack",
    "D10": "Crack",
    "D20": "Crack",
    "D40": "Pothole",
}

SOURCE_PRIORITY = {
    "RDD2022": 1,
    "SDNET2018": 2,
    "Kenyan_Data": 3,
}

print("Target classes:", TARGET_CLASSES)

print("\nDataset paths:")
print("RDD selected images:", RDD_SELECTED_IMAGES)
print("RDD selected labels:", RDD_SELECTED_LABELS)
print("SDNET annotated:", SDNET_ANNOTATED)
print("Kenyan annotated:", KENYAN_ANNOTATED)

Target classes: {1: 'Crack', 2: 'Pothole', 3: 'Surface deterioration'}

Dataset paths:
RDD selected images: /content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/images
RDD selected labels: /content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels
SDNET annotated: /content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/SDNET2018
Kenyan annotated: /content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/Kenyan_Data


## 1. Annotation Parsers

The project currently has two annotation situations:

- RDD2022: YOLO `.txt`
- Manually annotated SDNET2018/Kenyan_Data: Pascal VOC `.xml`

Do not force one format into another until the information has been validated.

In [2]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG", ".WEBP"}

def image_for_annotation(annotation_path):
    """
    Find the image corresponding to an RDD YOLO annotation.

    Expected selected RDD structure:

    Selected Data/RDD2022/
    ├── images/
    └── labels/
    """

    annotation_path = Path(annotation_path)

    for ext in IMAGE_EXTENSIONS:
        candidate = (
            RDD_SELECTED_IMAGES /
            f"{annotation_path.stem}{ext}"
        )

        if candidate.exists():
            return candidate

    return None

def parse_yolo_file(txt_path):
    objects = []
    errors = []

    for line_no, line in enumerate(
        open(txt_path, "r", encoding="utf-8", errors="ignore"),
        start=1
    ):
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            errors.append(f"Line {line_no}: expected 5 fields, got {len(parts)}")
            continue

        try:
            class_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
        except ValueError:
            errors.append(f"Line {line_no}: non-numeric value")
            continue

        objects.append({
            "class_id": class_id,
            "xc": xc,
            "yc": yc,
            "width": w,
            "height": h,
        })

    return objects, errors

def parse_voc_file(xml_path):
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    if size is None:
        raise ValueError("VOC file has no <size> element")

    width = int(size.findtext("width", "0"))
    height = int(size.findtext("height", "0"))

    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name", "").strip()

        bbox = obj.find("bndbox")
        if bbox is None:
            continue

        xmin = float(bbox.findtext("xmin", "0"))
        ymin = float(bbox.findtext("ymin", "0"))
        xmax = float(bbox.findtext("xmax", "0"))
        ymax = float(bbox.findtext("ymax", "0"))

        objects.append({
            "source_class": name,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "width": width,
            "height": height,
        })

    return width, height, objects

In [3]:
# ---------------------------------------------------------
# Diagnose selected RDD paths
# ---------------------------------------------------------

print("RDD selected directory:")
print(RDD_SELECTED_DIR)

print("\nRDD images directory:")
print(RDD_SELECTED_IMAGES)
print("Exists:", RDD_SELECTED_IMAGES.exists())

print("\nRDD labels directory:")
print(RDD_SELECTED_LABELS)
print("Exists:", RDD_SELECTED_LABELS.exists())

print("\nImages found:")
print(len(list(RDD_SELECTED_IMAGES.rglob("*"))))

print("\nTXT labels found:")
print(len(list(RDD_SELECTED_LABELS.rglob("*.txt"))))

print("\nContents of RDD selected directory:")
if RDD_SELECTED_DIR.exists():
    for item in RDD_SELECTED_DIR.iterdir():
        print(item.name)

RDD selected directory:
/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022

RDD images directory:
/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/images
Exists: True

RDD labels directory:
/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels
Exists: True

Images found:
1600

TXT labels found:
1600

Contents of RDD selected directory:
images
labels


## 2. Validate RDD2022 YOLO Annotations

In [4]:
def yolo_to_normalized_box(xc, yc, w, h):
    xmin = xc - w / 2
    ymin = yc - h / 2
    xmax = xc + w / 2
    ymax = yc + h / 2

    return xmin, ymin, xmax, ymax


rdd_records = []
rdd_errors = []
rdd_class_counts = Counter()

# ---------------------------------------------------------
# Validate only the selected RDD2022 labels
# ---------------------------------------------------------

rdd_txt_files = list(
    RDD_SELECTED_LABELS.rglob("*.txt")
)

print(
    "RDD selected label files found:",
    len(rdd_txt_files)
)


for txt_path in rdd_txt_files:

    image_path = image_for_annotation(
        txt_path
    )

    if image_path is None:
        rdd_errors.append({
            "file": str(txt_path),
            "error": "No matching selected image found"
        })
        continue

    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as exc:
        rdd_errors.append({
            "file": str(image_path),
            "error": f"Unreadable image: {exc}"
        })
        continue

    objects, parse_errors = parse_yolo_file(
        txt_path
    )

    for err in parse_errors:
        rdd_errors.append({
            "file": str(txt_path),
            "error": err
        })

    normalized_objects = []

    for obj in objects:

        class_id = obj["class_id"]

        if class_id not in RDD_CLASS_ID_TO_CODE:
            rdd_errors.append({
                "file": str(txt_path),
                "error": f"Unknown RDD class ID: {class_id}"
            })
            continue

        source_code = (
            RDD_CLASS_ID_TO_CODE[class_id]
        )

        # -------------------------------------------------
        # Ignore RDD class 4 / OTHER
        # -------------------------------------------------

        if source_code == "OTHER":
            continue

        if source_code not in RDD_CODE_TO_TARGET:
            continue

        if not (
            0 <= obj["xc"] <= 1
            and 0 <= obj["yc"] <= 1
            and 0 < obj["width"] <= 1
            and 0 < obj["height"] <= 1
        ):
            rdd_errors.append({
                "file": str(txt_path),
                "error": (
                    "Invalid normalized coordinates: "
                    f"{obj}"
                )
            })
            continue

        xmin, ymin, xmax, ymax = (
            yolo_to_normalized_box(
                obj["xc"],
                obj["yc"],
                obj["width"],
                obj["height"]
            )
        )

        if not (
            0 <= xmin < xmax <= 1
            and 0 <= ymin < ymax <= 1
        ):
            rdd_errors.append({
                "file": str(txt_path),
                "error": (
                    "Bounding box outside [0,1]: "
                    f"{obj}"
                )
            })
            continue

        target_class = (
            RDD_CODE_TO_TARGET[source_code]
        )

        rdd_class_counts[
            target_class
        ] += 1

        normalized_objects.append({
            "class_name": target_class,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "source_class": source_code,
        })

    if normalized_objects:
        rdd_records.append({
            "dataset": "RDD2022",
            "image_path": str(image_path),
            "annotation_path": str(txt_path),
            "annotation_format": "YOLO",
            "objects": normalized_objects,
        })


print(
    "\nValid RDD records:",
    len(rdd_records)
)

print(
    "RDD errors:",
    len(rdd_errors)
)

print(
    "RDD target-class counts:",
    dict(rdd_class_counts)
)

RDD selected label files found: 1600

Valid RDD records: 1492
RDD errors: 194
RDD target-class counts: {'Pothole': 1069, 'Crack': 1642}


In [5]:
# ---------------------------------------------------------
# Quick RDD error summary
# ---------------------------------------------------------

error_types = Counter()

for error in rdd_errors:
    message = error["error"]

    if "No matching selected image found" in message:
        error_types["Missing matching image"] += 1

    elif "Unreadable image" in message:
        error_types["Unreadable image"] += 1

    elif "Unknown RDD class ID" in message:
        error_types["Unknown class"] += 1

    elif "Invalid normalized coordinates" in message:
        error_types["Invalid YOLO coordinates"] += 1

    elif "Bounding box outside" in message:
        error_types["Bounding box outside image"] += 1

    else:
        error_types["Other parsing error"] += 1


print("RDD error summary:")

for error_type, count in error_types.items():
    print(f"{error_type}: {count}")


print("\nExample errors:")

for error in rdd_errors[:10]:
    print(error)

RDD error summary:
Bounding box outside image: 194

Example errors:
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/India_008339.txt', 'error': "Bounding box outside [0,1]: {'class_id': 3, 'xc': 0.547917, 'yc': 0.884028, 'width': 0.904167, 'height': 0.193056}"}
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/India_008338.txt', 'error': "Bounding box outside [0,1]: {'class_id': 3, 'xc': 0.289583, 'yc': 0.968056, 'width': 0.054167, 'height': 0.063889}"}
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/Japan_000217.txt', 'error': "Bounding box outside [0,1]: {'class_id': 3, 'xc': 0.579167, 'yc': 0.805, 'width': 0.841667, 'height': 0.173333}"}
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/Japan_000325.txt', 'error': "Bounding box outside [0,1]: {'class_id': 2, 'xc': 0.864167, 'yc': 0.873333, 'width': 0.271667, 'height': 0.253333}"}
{'file': '/content/drive/My

## 3. Validate Pascal VOC Annotations

This section expects manually annotated images exported as Pascal VOC.

Expected folders:

```text
Annotated Data/
├── SDNET2018/
│   ├── images/
│   └── annotations/
└── Kenyan_Data/
    ├── images/
    └── annotations/
```

If your DigitalSreeni export uses different folder names, change the paths below only.

In [7]:
# ---------------------------------------------------------
# Diagnose SDNET and Kenyan annotated-data paths
# ---------------------------------------------------------

print("ANNOTATED_DIR:")
print(ANNOTATED_DIR)
print("Exists:", ANNOTATED_DIR.exists())


for dataset_name, dataset_root in [
    ("SDNET2018", SDNET_ANNOTATED),
    ("Kenyan_Data", KENYAN_ANNOTATED),
]:

    images_dir = dataset_root / "images"
    annotations_dir = dataset_root / "annotations"

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("Dataset root:")
    print(dataset_root)
    print("Exists:", dataset_root.exists())

    print("\nImages directory:")
    print(images_dir)
    print("Exists:", images_dir.exists())

    print("\nAnnotations directory:")
    print(annotations_dir)
    print("Exists:", annotations_dir.exists())

    image_files = []
    if images_dir.exists():
        image_files = [
            p for p in images_dir.rglob("*")
            if p.is_file()
        ]

    xml_files = []
    if annotations_dir.exists():
        xml_files = list(
            annotations_dir.rglob("*.xml")
        )

    print("\nFiles inside images:", len(image_files))
    print("XML files inside annotations:", len(xml_files))

    if dataset_root.exists():
        print("\nFolders/files directly inside dataset root:")
        for item in dataset_root.iterdir():
            print(" -", item.name)

ANNOTATED_DIR:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data
Exists: True

SDNET2018
Dataset root:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/SDNET2018
Exists: True

Images directory:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/SDNET2018/images
Exists: True

Annotations directory:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/SDNET2018/annotations
Exists: True

Files inside images: 587
XML files inside annotations: 587

Folders/files directly inside dataset root:
 - images
 - annotations

Kenyan_Data
Dataset root:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/Kenyan_Data
Exists: True

Images directory:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/Kenyan_Data/images
Exists: True

Annotations directory:
/content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/Kenyan_Data/annotations
Exists: True

Files inside images: 278
XML files inside annotations: 278

Folders/files directly inside dataset root:
 - images
 - annot

In [9]:
# ---------------------------------------------------------
# Inspect one SDNET and one Kenyan XML file
# ---------------------------------------------------------

sdnet_sample_xml = next(
    (SDNET_ANNOTATED / "annotations").rglob("*.xml")
)

kenyan_sample_xml = next(
    (KENYAN_ANNOTATED / "annotations").rglob("*.xml")
)

for dataset_name, xml_path in [
    ("SDNET2018", sdnet_sample_xml),
    ("Kenyan_Data", kenyan_sample_xml),
]:

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)
    print("XML:", xml_path)

    root = ET.parse(xml_path).getroot()

    print("Root tag:", root.tag)

    print("\nDirect child tags:")
    for child in root:
        print(child.tag)

    print("\nNumber of direct <object> elements:")
    print(len(root.findall("object")))

    print("\nNumber of any nested <object> elements:")
    print(len(root.findall(".//object")))

    print("\nFirst 60 lines of XML:")
    with open(
        xml_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:
        for i, line in enumerate(f):
            if i >= 60:
                break
            print(line.rstrip())


SDNET2018
XML: /content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/SDNET2018/annotations/001-104.xml
Root tag: annotation

Direct child tags:
folder
filename
path
size
segmented
object

Number of direct <object> elements:
1

Number of any nested <object> elements:
1

First 60 lines of XML:
<?xml version="1.0" ?>
<annotation>
    <folder>images</folder>
    <filename>001-104.jpg</filename>
    <path>images\001-104.jpg</path>
    <size>
        <width>256</width>
        <height>256</height>
        <depth>3</depth>
    </size>
    <segmented>0</segmented>
    <object>
        <name>crack</name>
        <pose>Unspecified</pose>
        <truncated>0</truncated>
        <difficult>0</difficult>
    </object>
</annotation>

Kenyan_Data
XML: /content/drive/MyDrive/CS PROJECT II/Data/Annotated Data/Kenyan_Data/annotations/20260826_100802.xml
Root tag: annotation

Direct child tags:
folder
filename
path
size
segmented
object

Number of direct <object> elements:
1

Number of any nested <o

In [10]:
# ---------------------------------------------------------
# Check whether exported VOC XML files contain bounding boxes
# ---------------------------------------------------------

def check_bbox_export(dataset_name, annotation_dir):

    xml_files = list(
        Path(annotation_dir).rglob("*.xml")
    )

    files_with_objects = 0
    files_with_bboxes = 0
    files_without_bboxes = 0

    total_objects = 0
    total_bboxes = 0

    examples_without_bbox = []

    for xml_path in xml_files:

        root = ET.parse(xml_path).getroot()

        objects = root.findall("object")

        if objects:
            files_with_objects += 1

        file_has_bbox = False

        for obj in objects:

            total_objects += 1

            bbox = obj.find("bndbox")

            if bbox is not None:
                total_bboxes += 1
                file_has_bbox = True

        if objects and file_has_bbox:
            files_with_bboxes += 1

        elif objects:
            files_without_bboxes += 1

            if len(examples_without_bbox) < 5:
                examples_without_bbox.append(
                    xml_path.name
                )


    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("XML files:", len(xml_files))
    print("Files containing objects:", files_with_objects)
    print("Files containing bounding boxes:", files_with_bboxes)
    print("Files with objects but NO bounding boxes:", files_without_bboxes)

    print("Total objects:", total_objects)
    print("Total bounding boxes:", total_bboxes)

    if examples_without_bbox:
        print(
            "\nExamples without bounding boxes:",
            examples_without_bbox
        )


check_bbox_export(
    "SDNET2018",
    SDNET_ANNOTATED / "annotations"
)

check_bbox_export(
    "Kenyan_Data",
    KENYAN_ANNOTATED / "annotations"
)


SDNET2018
XML files: 587
Files containing objects: 587
Files containing bounding boxes: 0
Files with objects but NO bounding boxes: 587
Total objects: 662
Total bounding boxes: 0

Examples without bounding boxes: ['001-104.xml', '001-105.xml', '001-106.xml', '001-126.xml', '001-145.xml']

Kenyan_Data
XML files: 278
Files containing objects: 278
Files containing bounding boxes: 0
Files with objects but NO bounding boxes: 278
Total objects: 529
Total bounding boxes: 0

Examples without bounding boxes: ['20260826_100802.xml', '20260826_100804.xml', '20260826_100815.xml', '20260826_100826.xml', '20260826_101033(1).xml']


In [8]:
VOC_CLASS_MAP = {
    "crack": "Crack",
    "cracks": "Crack",
    "pothole": "Pothole",
    "potholes": "Pothole",
    "surface deterioration": "Surface deterioration",
    "surface_deterioration": "Surface deterioration",
    "surface deterioration ": "Surface deterioration",
}

def find_voc_image(xml_path, image_dir):
    xml_path = Path(xml_path)
    image_dir = Path(image_dir)

    for ext in IMAGE_EXTENSIONS:
        candidate = image_dir / f"{xml_path.stem}{ext}"
        if candidate.exists():
            return candidate

    return None

def validate_voc_dataset(dataset_name, annotation_dir, image_dir):
    records = []
    errors = []

    annotation_dir = Path(annotation_dir)
    image_dir = Path(image_dir)

    xml_files = list(annotation_dir.rglob("*.xml"))

    for xml_path in xml_files:
        image_path = find_voc_image(xml_path, image_dir)

        if image_path is None:
            errors.append({"file": str(xml_path), "error": "Matching image not found"})
            continue

        try:
            width, height, objects = parse_voc_file(xml_path)
        except Exception as exc:
            errors.append({"file": str(xml_path), "error": str(exc)})
            continue

        if width <= 0 or height <= 0:
            errors.append({"file": str(xml_path), "error": "Invalid image dimensions"})
            continue

        normalized_objects = []

        for obj in objects:
            raw_name = obj["source_class"].strip().lower()

            if raw_name not in VOC_CLASS_MAP:
                errors.append({
                    "file": str(xml_path),
                    "error": f"Unknown class name: {obj['source_class']}"
                })
                continue

            xmin = obj["xmin"] / width
            ymin = obj["ymin"] / height
            xmax = obj["xmax"] / width
            ymax = obj["ymax"] / height

            if not (0 <= xmin < xmax <= 1 and 0 <= ymin < ymax <= 1):
                errors.append({
                    "file": str(xml_path),
                    "error": f"Invalid bounding box: {obj}"
                })
                continue

            target_class = VOC_CLASS_MAP[raw_name]

            normalized_objects.append({
                "class_name": target_class,
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
                "source_class": obj["source_class"],
            })

        if normalized_objects:
            records.append({
                "dataset": dataset_name,
                "image_path": str(image_path),
                "annotation_path": str(xml_path),
                "annotation_format": "PascalVOC",
                "objects": normalized_objects,
            })

    return records, errors

sdnet_records, sdnet_errors = validate_voc_dataset(
    "SDNET2018",
    SDNET_ANNOTATED / "annotations",
    SDNET_ANNOTATED / "images"
)

kenyan_records, kenyan_errors = validate_voc_dataset(
    "Kenyan_Data",
    KENYAN_ANNOTATED / "annotations",
    KENYAN_ANNOTATED / "images"
)

print("SDNET valid records:", len(sdnet_records), "errors:", len(sdnet_errors))
print("Kenyan valid records:", len(kenyan_records), "errors:", len(kenyan_errors))

SDNET valid records: 0 errors: 0
Kenyan valid records: 0 errors: 0


...

In [12]:
# ---------------------------------------------------------
# Recover SDNET and Kenyan bounding boxes from mask images
# ---------------------------------------------------------

MASK_CLASS_MAP = {
    "crack": "Crack",
    "pothole": "Pothole",
    "surface deterioration": "Surface deterioration",
    "surface_deterioration": "Surface deterioration",
}


def find_original_image(mask_path, image_dir):
    """
    Match a mask filename such as:
        001-104_crack_mask.png
    back to:
        001-104.jpg
    """

    mask_path = Path(mask_path)
    image_dir = Path(image_dir)

    stem = mask_path.stem

    # Remove expected mask suffixes.
    for suffix in [
        "_crack_mask",
        "_pothole_mask",
        "_surface deterioration_mask",
        "_surface_deterioration_mask",
        "_mask",
    ]:
        if stem.lower().endswith(suffix.lower()):
            stem = stem[: -len(suffix)]
            break

    for ext in IMAGE_EXTENSIONS:
        candidate = image_dir / f"{stem}{ext}"

        if candidate.exists():
            return candidate

    return None


def boxes_from_mask(mask_path, min_area=1):
    """
    Convert a binary mask into one or more bounding boxes.

    Each connected component becomes a separate bounding box.
    """

    mask = np.array(
        Image.open(mask_path)
    )

    # Some masks may have multiple channels.
    if mask.ndim == 3:
        mask = np.any(mask > 0, axis=2).astype(np.uint8)
    else:
        mask = (mask > 0).astype(np.uint8)

    if mask.max() == 0:
        return []

    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    boxes = []

    for component_id in range(
        1,
        num_labels
    ):
        x, y, w, h, area = (
            stats[component_id]
        )

        if area < min_area:
            continue

        boxes.append({
            "xmin": int(x),
            "ymin": int(y),
            "xmax": int(x + w),
            "ymax": int(y + h),
            "area": int(area),
        })

    return boxes


def validate_mask_dataset(
    dataset_name,
    image_dir,
    mask_root
):
    """
    Build normalized object-detection records
    from class-specific mask folders.
    """

    records_by_image = {}
    errors = []

    image_dir = Path(image_dir)
    mask_root = Path(mask_root)

    if not mask_root.exists():
        print(
            f"{dataset_name}: mask folder not found:"
        )
        print(mask_root)

        return [], [{
            "file": str(mask_root),
            "error": "Mask directory not found"
        }]

    for class_dir in mask_root.iterdir():

        if not class_dir.is_dir():
            continue

        raw_class = (
            class_dir.name
            .strip()
            .lower()
        )

        if raw_class not in MASK_CLASS_MAP:

            errors.append({
                "file": str(class_dir),
                "error": (
                    f"Unknown mask class folder: "
                    f"{class_dir.name}"
                )
            })

            continue

        target_class = (
            MASK_CLASS_MAP[raw_class]
        )

        mask_files = [
            p
            for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower()
            in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
        ]

        for mask_path in mask_files:

            image_path = find_original_image(
                mask_path,
                image_dir
            )

            if image_path is None:

                errors.append({
                    "file": str(mask_path),
                    "error": "Matching original image not found"
                })

                continue

            try:
                with Image.open(image_path) as img:
                    width, height = img.size

            except Exception as exc:

                errors.append({
                    "file": str(image_path),
                    "error": (
                        f"Unreadable image: {exc}"
                    )
                })

                continue

            boxes = boxes_from_mask(
                mask_path
            )

            if not boxes:

                errors.append({
                    "file": str(mask_path),
                    "error": "Mask contains no positive pixels"
                })

                continue

            image_key = str(image_path)

            if image_key not in records_by_image:

                records_by_image[image_key] = {
                    "dataset": dataset_name,
                    "image_path": str(image_path),
                    "annotation_path": [],
                    "annotation_format": "Mask-derived",
                    "objects": [],
                }

            records_by_image[
                image_key
            ]["annotation_path"].append(
                str(mask_path)
            )

            for box in boxes:

                xmin = box["xmin"] / width
                ymin = box["ymin"] / height
                xmax = box["xmax"] / width
                ymax = box["ymax"] / height

                if not (
                    0 <= xmin < xmax <= 1
                    and 0 <= ymin < ymax <= 1
                ):

                    errors.append({
                        "file": str(mask_path),
                        "error": (
                            f"Invalid mask-derived bbox: "
                            f"{box}"
                        )
                    })

                    continue

                records_by_image[
                    image_key
                ]["objects"].append({
                    "class_name": target_class,
                    "xmin": xmin,
                    "ymin": ymin,
                    "xmax": xmax,
                    "ymax": ymax,
                    "source_class": class_dir.name,
                })

    records = [
        record
        for record
        in records_by_image.values()
        if record["objects"]
    ]

    return records, errors

In [15]:
# ---------------------------------------------------------
# Build SDNET and Kenyan records from masks
# ---------------------------------------------------------

sdnet_records, sdnet_errors = (
    validate_mask_dataset(
        "SDNET2018",
        SDNET_ANNOTATED / "images",
        SDNET_MASK_DIR
    )
)

kenyan_records, kenyan_errors = (
    validate_mask_dataset(
        "Kenyan_Data",
        KENYAN_ANNOTATED / "images",
        KENYAN_MASK_DIR
    )
)


print(
    "SDNET valid records:",
    len(sdnet_records),
    "errors:",
    len(sdnet_errors)
)

print(
    "Kenyan valid records:",
    len(kenyan_records),
    "errors:",
    len(kenyan_errors)
)


sdnet_object_count = sum(
    len(r["objects"])
    for r in sdnet_records
)

kenyan_object_count = sum(
    len(r["objects"])
    for r in kenyan_records
)

print(
    "SDNET objects:",
    sdnet_object_count
)

print(
    "Kenyan objects:",
    kenyan_object_count
)

SDNET valid records: 587 errors: 0
Kenyan valid records: 278 errors: 0
SDNET objects: 622
Kenyan objects: 446


In [16]:
# ---------------------------------------------------------
# Mask / original-image pairing validation
# ---------------------------------------------------------

def check_mask_pairing(
    dataset_name,
    image_dir,
    mask_root
):

    image_dir = Path(image_dir)
    mask_root = Path(mask_root)

    images = [
        p
        for p in image_dir.rglob("*")
        if p.is_file()
        and p.suffix in IMAGE_EXTENSIONS
    ]

    image_stems = {
        p.stem
        for p in images
    }

    matched_masks = 0
    unmatched_masks = []

    for class_dir in mask_root.iterdir():

        if not class_dir.is_dir():
            continue

        for mask_path in class_dir.rglob("*"):

            if not mask_path.is_file():
                continue

            if mask_path.suffix.lower() not in {
                ".png",
                ".jpg",
                ".jpeg",
                ".tif",
                ".tiff",
            }:
                continue

            image_path = find_original_image(
                mask_path,
                image_dir
            )

            if image_path is None:

                unmatched_masks.append(
                    mask_path.name
                )

            else:
                matched_masks += 1

    print(f"\n{dataset_name}")
    print("Original images:", len(images))
    print("Matched masks:", matched_masks)
    print(
        "Unmatched masks:",
        len(unmatched_masks)
    )

    if unmatched_masks:
        print(
            "Examples:",
            unmatched_masks[:10]
        )


check_mask_pairing(
    "SDNET2018",
    SDNET_ANNOTATED / "images",
    SDNET_MASK_DIR
)

check_mask_pairing(
    "Kenyan_Data",
    KENYAN_ANNOTATED / "images",
    KENYAN_MASK_DIR
)


SDNET2018
Original images: 587
Matched masks: 587
Unmatched masks: 0

Kenyan_Data
Original images: 278
Matched masks: 313
Unmatched masks: 0


In [17]:
# ---------------------------------------------------------
# Exact duplicate image check
# Kenyan_Data + SDNET2018
# ---------------------------------------------------------

def file_md5(path, chunk_size=1024 * 1024):

    md5 = hashlib.md5()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


def check_exact_duplicates(
    dataset_name,
    image_paths
):

    hash_records = []

    for image_path in image_paths:

        image_hash = file_md5(
            image_path
        )

        hash_records.append({
            "image_path": str(image_path),
            "filename": image_path.name,
            "hash": image_hash,
        })

    hash_df = pd.DataFrame(
        hash_records
    )

    duplicate_hashes = (
        hash_df[
            hash_df.duplicated(
                "hash",
                keep=False
            )
        ]
        .sort_values("hash")
    )

    print(f"\n{dataset_name}")
    print(
        "Images checked:",
        len(hash_df)
    )

    print(
        "Unique image hashes:",
        hash_df["hash"].nunique()
    )

    print(
        "Images involved in exact duplicates:",
        len(duplicate_hashes)
    )

    if not duplicate_hashes.empty:
        display(
            duplicate_hashes
        )


kenyan_images = [
    p
    for p in (KENYAN_ANNOTATED / "images").rglob("*")
    if p.is_file()
    and p.suffix in IMAGE_EXTENSIONS
]

sdnet_images = [
    p
    for p in (SDNET_ANNOTATED / "images").rglob("*")
    if p.is_file()
    and p.suffix in IMAGE_EXTENSIONS
]

check_exact_duplicates(
    "Kenyan_Data",
    kenyan_images
)

check_exact_duplicates(
    "SDNET2018",
    sdnet_images
)


Kenyan_Data
Images checked: 278
Unique image hashes: 276
Images involved in exact duplicates: 4


,image_path,filename,hash
156,/content/drive/MyDrive/CS PROJECT II/Data/Anno...,20260919_130504.jpg,7b2c1268689aabd6457fb8ff992f4d46
161,/content/drive/MyDrive/CS PROJECT II/Data/Anno...,20260919_130504(1).jpg,7b2c1268689aabd6457fb8ff992f4d46
135,/content/drive/MyDrive/CS PROJECT II/Data/Anno...,20260917_114030.jpg,97edffc911a18de8d56c1a35ce2887db
211,/content/drive/MyDrive/CS PROJECT II/Data/Anno...,20260917_114030(1).jpg,97edffc911a18de8d56c1a35ce2887db



SDNET2018
Images checked: 587
Unique image hashes: 587
Images involved in exact duplicates: 0


## 4. Important SDNET2018 Note

Do **not** automatically assign a full-image bounding box to every SDNET2018 crack patch.

For this project, selected SDNET images that are being used for object detection should have the visible defect manually bounded where practical. A full-image box would teach the detector that the entire patch is the defect and would provide weak localization evidence.

If the selected SDNET data has not yet been manually annotated, stop here and complete the required annotation before using those images in training.

In [18]:
all_records = rdd_records + sdnet_records + kenyan_records

records_df = pd.DataFrame([
    {
        "dataset": r["dataset"],
        "image_path": r["image_path"],
        "annotation_path": r["annotation_path"],
        "annotation_format": r["annotation_format"],
        "object_count": len(r["objects"]),
        "classes": sorted(set(o["class_name"] for o in r["objects"]))
    }
    for r in all_records
])

display(records_df.groupby(["dataset", "annotation_format"]).size().reset_index(name="records"))

all_class_counts = Counter()
for r in all_records:
    for obj in r["objects"]:
        all_class_counts[obj["class_name"]] += 1

print("Unified object counts:")
print(dict(all_class_counts))

,dataset,annotation_format,records
0,Kenyan_Data,Mask-derived,278
1,RDD2022,YOLO,1492
2,SDNET2018,Mask-derived,587


Unified object counts:
{'Pothole': 1231, 'Crack': 2493, 'Surface deterioration': 55}


## 5. Create a Normalized Manifest

The manifest is the hand-off between annotation validation and TFRecord generation.

It contains:
- image path
- annotation source
- normalized bounding boxes
- target class names

The Raw Data is not modified.

In [19]:
MANIFEST_PATH = PROCESSED_ANNOTATIONS / "validated_manifest.jsonl"

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    for record in all_records:
        f.write(json.dumps(record) + "\n")

print("Manifest written to:", MANIFEST_PATH)
print("Records:", len(all_records))

Manifest written to: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/annotations/validated_manifest.jsonl
Records: 2357


## 6. Train / Validation / Test Split

The split is performed at the image level so an image cannot appear in more than one split.

For Kenyan_Data, retain a local test portion so the final evaluation can report performance on locally sourced images.

If you have multiple photographs of the same physical location/site, those images should ideally be grouped before splitting to prevent location leakage.

In [20]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42


# ---------------------------------------------------------
# Load validated records
# ---------------------------------------------------------

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


# ---------------------------------------------------------
# Separate records by source dataset
# ---------------------------------------------------------

rdd = [
    r
    for r in records
    if r["dataset"] == "RDD2022"
]

sdnet = [
    r
    for r in records
    if r["dataset"] == "SDNET2018"
]

kenyan = [
    r
    for r in records
    if r["dataset"] == "Kenyan_Data"
]


print("RDD2022 records:", len(rdd))
print("SDNET2018 records:", len(sdnet))
print("Kenyan_Data records:", len(kenyan))


# ---------------------------------------------------------
# Reproducible 70 / 15 / 15 split
# ---------------------------------------------------------

def split_70_15_15(items, seed=42):

    if len(items) < 3:
        return items, [], []

    # First reserve 15% for testing.
    train_val, test = train_test_split(
        items,
        test_size=0.15,
        random_state=seed,
        shuffle=True
    )

    # train_val contains 85% of the original dataset.
    # To obtain 15% of the ORIGINAL dataset for validation:
    #
    # 15 / 85 = 0.176470588...
    validation_fraction = 0.15 / 0.85

    train, validation = train_test_split(
        train_val,
        test_size=validation_fraction,
        random_state=seed,
        shuffle=True
    )

    return train, validation, test


# ---------------------------------------------------------
# Split EACH dataset independently
# ---------------------------------------------------------

train_rdd, val_rdd, test_rdd = split_70_15_15(
    rdd,
    seed=RANDOM_SEED
)

train_sdnet, val_sdnet, test_sdnet = split_70_15_15(
    sdnet,
    seed=RANDOM_SEED
)

train_ke, val_ke, test_ke = split_70_15_15(
    kenyan,
    seed=RANDOM_SEED
)


# ---------------------------------------------------------
# Combine corresponding partitions
# ---------------------------------------------------------

train_records = (
    train_rdd
    + train_sdnet
    + train_ke
)

val_records = (
    val_rdd
    + val_sdnet
    + val_ke
)

test_records = (
    test_rdd
    + test_sdnet
    + test_ke
)


# ---------------------------------------------------------
# Deterministic shuffle after combining
# ---------------------------------------------------------

random.Random(
    RANDOM_SEED
).shuffle(train_records)

random.Random(
    RANDOM_SEED
).shuffle(val_records)

random.Random(
    RANDOM_SEED
).shuffle(test_records)


# ---------------------------------------------------------
# Display split sizes
# ---------------------------------------------------------

print("\nRDD2022 split:")
print("Train:", len(train_rdd))
print("Validation:", len(val_rdd))
print("Test:", len(test_rdd))

print("\nSDNET2018 split:")
print("Train:", len(train_sdnet))
print("Validation:", len(val_sdnet))
print("Test:", len(test_sdnet))

print("\nKenyan_Data split:")
print("Train:", len(train_ke))
print("Validation:", len(val_ke))
print("Test:", len(test_ke))

print("\nCombined split:")
print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

RDD2022 records: 1492
SDNET2018 records: 587
Kenyan_Data records: 278

RDD2022 split:
Train: 1044
Validation: 224
Test: 224

SDNET2018 split:
Train: 410
Validation: 88
Test: 89

Kenyan_Data split:
Train: 194
Validation: 42
Test: 42

Combined split:
Train: 1648
Validation: 354
Test: 355


In [21]:
# ---------------------------------------------------------
# Split leakage check
# ---------------------------------------------------------

train_paths = {
    r["image_path"]
    for r in train_records
}

val_paths = {
    r["image_path"]
    for r in val_records
}

test_paths = {
    r["image_path"]
    for r in test_records
}


train_val_overlap = (
    train_paths & val_paths
)

train_test_overlap = (
    train_paths & test_paths
)

val_test_overlap = (
    val_paths & test_paths
)


print(
    "Train / Validation overlap:",
    len(train_val_overlap)
)

print(
    "Train / Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation / Test overlap:",
    len(val_test_overlap)
)


assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0


print(
    "\nImage-level leakage check passed."
)

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

Image-level leakage check passed.


In [22]:
SPLIT_DIR = PROCESSED_ANNOTATIONS / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_records_list in {
    "train": train_records,
    "validation": val_records,
    "test": test_records,
}.items():
    path = SPLIT_DIR / f"{split_name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for record in split_records_list:
            f.write(json.dumps(record) + "\n")
    print(split_name, "->", path)

train -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/annotations/splits/train.jsonl
validation -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/annotations/splits/validation.jsonl
test -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/annotations/splits/test.jsonl


In [23]:
# ---------------------------------------------------------
# Dataset-source distribution across splits
# ---------------------------------------------------------

def dataset_counts(records):
    return Counter(
        r["dataset"]
        for r in records
    )


print("Train datasets:")
print(dict(dataset_counts(train_records)))

print("\nValidation datasets:")
print(dict(dataset_counts(val_records)))

print("\nTest datasets:")
print(dict(dataset_counts(test_records)))


# ---------------------------------------------------------
# Verify approximate 70 / 15 / 15 percentages
# ---------------------------------------------------------

print("\nDataset split percentages:")

for dataset_name, train_set, val_set, test_set in [
    (
        "RDD2022",
        train_rdd,
        val_rdd,
        test_rdd
    ),
    (
        "SDNET2018",
        train_sdnet,
        val_sdnet,
        test_sdnet
    ),
    (
        "Kenyan_Data",
        train_ke,
        val_ke,
        test_ke
    ),
]:

    total = (
        len(train_set)
        + len(val_set)
        + len(test_set)
    )

    if total == 0:
        print(
            f"\n{dataset_name}: no records"
        )
        continue

    train_pct = (
        len(train_set) / total
    ) * 100

    val_pct = (
        len(val_set) / total
    ) * 100

    test_pct = (
        len(test_set) / total
    ) * 100

    print(f"\n{dataset_name}")
    print(
        f"Train: {len(train_set)} "
        f"({train_pct:.2f}%)"
    )
    print(
        f"Validation: {len(val_set)} "
        f"({val_pct:.2f}%)"
    )
    print(
        f"Test: {len(test_set)} "
        f"({test_pct:.2f}%)"
    )

Train datasets:
{'SDNET2018': 410, 'RDD2022': 1044, 'Kenyan_Data': 194}

Validation datasets:
{'Kenyan_Data': 42, 'RDD2022': 224, 'SDNET2018': 88}

Test datasets:
{'Kenyan_Data': 42, 'RDD2022': 224, 'SDNET2018': 89}

Dataset split percentages:

RDD2022
Train: 1044 (69.97%)
Validation: 224 (15.01%)
Test: 224 (15.01%)

SDNET2018
Train: 410 (69.85%)
Validation: 88 (14.99%)
Test: 89 (15.16%)

Kenyan_Data
Train: 194 (69.78%)
Validation: 42 (15.11%)
Test: 42 (15.11%)


## 7. Final Annotation QC

Review the error lists before training.

If errors are substantial, fix the source annotations and rerun this notebook. Do not silently discard large portions of the dataset.

In [24]:
print("RDD errors:", len(rdd_errors))
print("SDNET errors:", len(sdnet_errors))
print("Kenyan errors:", len(kenyan_errors))

for title, errors in [
    ("RDD2022", rdd_errors),
    ("SDNET2018", sdnet_errors),
    ("Kenyan_Data", kenyan_errors),
]:
    if errors:
        print(f"\n{title} error examples:")
        for error in errors[:10]:
            print(error)

RDD errors: 194
SDNET errors: 0
Kenyan errors: 0

RDD2022 error examples:
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/India_008339.txt', 'error': "Bounding box outside [0,1]: {'class_id': 3, 'xc': 0.547917, 'yc': 0.884028, 'width': 0.904167, 'height': 0.193056}"}
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/India_008338.txt', 'error': "Bounding box outside [0,1]: {'class_id': 3, 'xc': 0.289583, 'yc': 0.968056, 'width': 0.054167, 'height': 0.063889}"}
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/Japan_000217.txt', 'error': "Bounding box outside [0,1]: {'class_id': 3, 'xc': 0.579167, 'yc': 0.805, 'width': 0.841667, 'height': 0.173333}"}
{'file': '/content/drive/MyDrive/CS PROJECT II/Data/Selected Data/RDD2022/labels/Japan_000325.txt', 'error': "Bounding box outside [0,1]: {'class_id': 2, 'xc': 0.864167, 'yc': 0.873333, 'width': 0.271667, 'height': 0.253333}"}
{'file': '/content/dr